# Codegen
CodeGen is a family of autoregressive language models for program synthesis from the paper:<br>
**Codegen: An open large language model for code with multi-turn program synthesis**


In this Notebook we are going to load the **codegen-2B-mono** model where:
* **2B** stands for the amount of parameters of the model
* **mono** stands on what is trained on, in this case mono means that the model was finetuned for python code generation only.

In [ ]:
# PIP INSTALLS
# These will eventually be already installed in the container, so there should be no pip install inside the notebook
!pip install accelerate
#Transformers needs to be updated
!pip install  transformers
#Install our qaic apis
!python3 -m pip install /opt/qti-aic/dev/lib/x86_64/qaic-0.0.1-py3-none-any.whl
!pip install torch
!pip install numpy
!pip install onnx

In [ ]:
!pip install tensorflow

## Imports

In [1]:
import qaic
import numpy as np
import tensorflow as tf
import os
import shutil
from pathlib import Path
import torch
from torch.onnx import export
import onnx
from packaging import version

2023-07-07 11:59:30.961235: I tensorflow/tsl/cuda/cudart_stub.cc:28] Could not find cuda drivers on your machine, GPU will not be used.
2023-07-07 11:59:32.632464: I tensorflow/tsl/cuda/cudart_stub.cc:28] Could not find cuda drivers on your machine, GPU will not be used.
2023-07-07 11:59:32.638910: I tensorflow/core/platform/cpu_feature_guard.cc:182] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2023-07-07 11:59:43.730753: W tensorflow/compiler/tf2tensorrt/utils/py_utils.cc:38] TF-TRT Warning: Could not find TensorRT


## Transformers from Huggingface

We are going to use the Transformer library from Hugging Face to obtain the model.

Transformers offers the CodeGenForCausalLM class for text generation using Codegen.<br>
Codegen is an autoregressive (or Causal) model, meaning that tokens from the previous steps are used to predict the token at the current step.

In [2]:
from transformers import CodeGenForCausalLM, AutoTokenizer

/local/mnt/workspace/fbuoncom/venv/lib/python3.8/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Download the model
The Pytorch model will be donwloaded using transformers and then saved locally to be loaded for ONNX export.

In [ ]:
# Load the model and tokenizer
checkpoint = "Salesforce/codegen-2B-mono"
model = CodeGenForCausalLM.from_pretrained(checkpoint)
tokenizer = AutoTokenizer.from_pretrained(checkpoint)

model.save_pretrained("./cd/model",max_shard_size='20GB') # So that the model is not split in 2
tokenizer.save_pretrained("./cd/tokenizer")
del model
del tokenizer

## Load the saved model
We are going to load the tokenizer only, the actual model will be loaded during ONNX conversion

In [3]:
tokenizer = AutoTokenizer.from_pretrained('./cd/tokenizer')

## First generation of the ONNX
Here we export the model to ONNX.
Exporting to ONNX is heavily documented online, and there are many scripts available.<br>
Here a basic export script is used.<br>

From the forward() method of the model CodeGenForCausalLM, we can understand what is the **input_ids** required by the model.

Thes input_idse are the result of the tokenizer component

```txt
input_ids (torch.LongTensor of shape (batch_size, sequence_length)):
Indices of input sequence tokens in the vocabular
```
           


In [4]:
@torch.no_grad()
def convert_models(model_path: str, output_path: str, opset: int):
    print("Loading the model")
    model = CodeGenForCausalLM.from_pretrained(model_path)
    output_path = Path(output_path)
    output_model_path = output_path / "model.onnx"
    print("Loaded the model")
    
    # Dicts
    input_names = ["input_ids"]
    output_names = ["output"]
    dynamic_axes = {
        "input_ids": {0: "batch_size", 1: "sequence_length"},
        "output": {0: "batch_size", 1: "sequence_length"},
    }
    
    print("Creating the dummy input")
    text = "def dummy_input:"
    # Make the pad token equal to the eos token
    tokenizer.pad_token = tokenizer.eos_token
    tokens = tokenizer(text,
                       padding='max_length',
                       max_length=128,
                       return_tensors="pt")
    
    print("Created the dummy input")
    print("Starting the export")
    torch.onnx.export(
        model,
        tokens['input_ids'],
        f=output_model_path.as_posix(),
        input_names=input_names,
        output_names=output_names,
        dynamic_axes=dynamic_axes,
        do_constant_folding=True,
        opset_version=opset
    )
    print("Finished the export")
    
    output_model_path = str(output_model_path.absolute().as_posix())
    output_model_dir = os.path.dirname(output_model_path)
    model = onnx.load(output_model_path)
    # clean up existing tensor files
    shutil.rmtree(output_model_dir)
    os.mkdir(output_model_dir)
    
    print("Started the squashing of the model")
    onnx.save_model(
        model,
        output_model_path,
        save_as_external_data=True,
        all_tensors_to_one_file=True,
        location="weights.pb",
        convert_attribute=False,
    )
    print("Squashed model")
    del model

In [5]:
!mkdir cd/model_onnx

mkdir: cannot create directory ‘cd/model_onnx’: File exists


In [6]:
# Convert the UNet with ONNX opset 14
convert_models('./cd/model','./cd/model_onnx',14)
print("Finished")

Loading the model
Loaded the model
Creating the dummy input
Created the dummy input
Starting the export


/local/mnt/workspace/fbuoncom/venv/lib/python3.8/site-packages/transformers/models/codegen/modeling_codegen.py:147: TracerWarning: torch.tensor results are registered as constants in the trace. You can safely ignore this warning if you use this function to create tensors out of constant variables that would be the same every time you call this function. In any other case, this might cause the trace to be incorrect.
  mask_value = torch.tensor(mask_value, dtype=attn_weights.dtype).to(attn_weights.device)


======== Diagnostic Run torch.onnx.export version 2.1.0.dev20230612+cpu ========
verbose: False, log level: 40
======================= 0 NONE 0 NOTE 0 WARNING 0 ERROR ========================

Finished the export
Started the squashing of the model
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
Squashed model
Finished


## Use of the HL Python API

We will use qaic.Session to compile our QPC.

Let's analyze the various parameters passed to the qaic.Session() constructor.

| Parameter | Explanation |
| --- | --- |
| **dev_id** | The device id we are going to use for our calculation|
| **aic_num_cores** | 	Number of aic cores to be used for inference |
| **convert_to_fp16** | Run all floating-point in fp16 |
| **onnx_define_symbol** | These comes from the dynamic axes of the onnx model that need to be "fixed" when compiling the model. These are normally the input dimensions that could be changed dynamically at runtime when using the ONNXRuntime. Here, since we compile them to a QPC, we need them to be fixed |

In [4]:
onnx_model_path="./cd/model_onnx/model.onnx"
qpc_model_path="./cd/model_qpc/programqpc.bin"

Here, we also define the sequence_length symbol to be 512.<br>
This value can range from 32 to 2048. 

In [8]:
!mkdir ./cd/model_qpc/

mkdir: cannot create directory ‘./cd/model_qpc/’: File exists


In [9]:
qaic_sess = qaic.Session(onnx_model_path,
                         dev_id=0,
                         aic_num_cores=14,
                         convert_to_fp16=True,
                         onnx_define_symbol={"batch_size":1,
                                             "sequence_length":512},
                         output_dir="./cd/model_qpc/"
                        )

Options yaml file available at:  ./cd/model_qpc//options.yaml


In [10]:
# Free memory, this is after the first compilation of the qpc
del qaic_sess

## Prepare our input text

Now we have our Session ready we can run inference, let's create an example prompt.<br>
Since we fixed our model input size to 512 we need to provide a padded input.<br>
To get the initial length of our string in tokens, so that we can add new token on top of it, we can run the tokenizer without padding.<br>
This will be useful when generating the new text.

In [5]:
text="# Class definition of a Dog"

In [6]:
tokens_no_pad = tokenizer(text,
                   return_tensors="np")['input_ids']
initial_sequence_length=int(tokens_no_pad.shape[1])

In [7]:
print(tokens_no_pad)
print(initial_sequence_length)

[[   2 5016 6770  286  257 8532]]
6


Here we generate the actual padded sentence

In [8]:
tokenizer.pad_token = tokenizer.eos_token
tokens = tokenizer(text,
                   padding='max_length',
                   max_length=512,
                   return_tensors="np")['input_ids']

In [9]:
print(tokens)

[[    2  5016  6770   286   257  8532 50256 50256 50256 50256 50256 50256
  50256 50256 50256 50256 50256 50256 50256 50256 50256 50256 50256 50256
  50256 50256 50256 50256 50256 50256 50256 50256 50256 50256 50256 50256
  50256 50256 50256 50256 50256 50256 50256 50256 50256 50256 50256 50256
  50256 50256 50256 50256 50256 50256 50256 50256 50256 50256 50256 50256
  50256 50256 50256 50256 50256 50256 50256 50256 50256 50256 50256 50256
  50256 50256 50256 50256 50256 50256 50256 50256 50256 50256 50256 50256
  50256 50256 50256 50256 50256 50256 50256 50256 50256 50256 50256 50256
  50256 50256 50256 50256 50256 50256 50256 50256 50256 50256 50256 50256
  50256 50256 50256 50256 50256 50256 50256 50256 50256 50256 50256 50256
  50256 50256 50256 50256 50256 50256 50256 50256 50256 50256 50256 50256
  50256 50256 50256 50256 50256 50256 50256 50256 50256 50256 50256 50256
  50256 50256 50256 50256 50256 50256 50256 50256 50256 50256 50256 50256
  50256 50256 50256 50256 50256 50256 

## Load the QPC on the card
We can create a session from the QPC we prevously compiled

In [10]:
qpc_model_path = "./cd/model_qpc/programqpc.bin"
yaml_options_path = "./cd/model_qpc/options.yaml"
qaic_sess = qaic.Session(qpc_model_path, options_path=yaml_options_path)

## Autoregressive Loop
This is a small example of a possible loop that takes the tokens and select each next token based on the most probable one.<br>
This loop is deterministic, it will produce the same result given the same prompt, it is called **greedy decoding**.

This is only one of the possible way to select the next token, like Beam Search, Temperature or Top-K Samplinging.

In [11]:
def generate_qaic_output(qaic_sess, original_input_ids, initial_token_size, max_new_tokens):
    cur_len = initial_token_size
    input_ids = original_input_ids

    while cur_len < max_new_tokens:
        logits = qaic_sess.run({"input_ids":input_ids})['output']
        logits = logits.reshape(1,512,51200)
        
        # Get the logits of the last token
        next_token_logits = logits[:, cur_len-1, :]
        del logits
        # Choose the token with the highest probability
        next_token = np.argmax(next_token_logits, axis=1)[:, np.newaxis]

        # Append next_token at the end of input_ids
        input_ids[0][cur_len]=next_token
        cur_len += 1
    
    return input_ids

A small example of **Top-K sampling** is also provided

In [4]:
from transformers import top_k_top_p_filtering
from torch.nn import functional as F
def generate_qaic_output_top_k(qaic_sess, original_input_ids, initial_token_size, max_new_tokens):
    cur_len = initial_token_size
    input_ids = original_input_ids

    while cur_len < max_new_tokens:
        logits = qaic_sess.run({"input_ids":input_ids})['output']
        logits = logits.reshape(1,512,51200)
        
        # Get the logits of the last token
        next_token_logits = logits[:, cur_len-1, :]

        # Keep top 30 logits at max; stop if cumulative probability >= 1.0.
        top_logits = top_k_top_p_filtering(torch.Tensor(next_token_logits), top_k=100, top_p=1.0)
        
        # Softmax the logits into probabilities
        probabilities = F.softmax(top_logits, dim=-1)
        
        # Generate next token
        next_token = torch.multinomial(probabilities, num_samples=1)
        
        # Append next_token at the end of input_ids
        input_ids[0][cur_len]=next_token
        cur_len += 1

    return input_ids

### Inference - Greedy Decoding 

In [30]:
MAX_LENGTH = 30

In [31]:
starting_inputs=tokens

In [32]:
output_tokens = generate_qaic_output(qaic_sess,starting_inputs,initial_sequence_length,MAX_LENGTH)

qaic: WARNING: Acitvating network, this will add up to time of first inference


We now have the output in token form, we are going to use the decode function of the tokenizer to get actual text

In [33]:
decoded_string = tokenizer.decode(output_tokens[0], skip_special_tokens=True)

In [34]:
print(decoded_string)

# Class definition of a Dog
#
# Author: David Fisher
# Course: CS151
# Date Created: 2020-02-20


In [35]:
qaic_sess.reset()

### Inference - Top-k Decoding

In [ ]:
for i in range(3):
    print(f"############################## Option {i+1} #############################################")
    output_tokens = generate_qaic_output_top_k(qaic_sess,starting_inputs,initial_sequence_length,MAX_LENGTH)
    decoded_string = tokenizer.decode(output_tokens[0], skip_special_tokens=True)
    print(decoded_string)
    print(f"#####################################################################################")

qaic: WARNING: Acitvating network, this will add up to time of first inference


############################## Option 1 #############################################
# Class definition of a Dog for the game Hunt the Wumpus
import tkinter

# import Image
import PIL.Image
#########################################################################################
############################## Option 2 #############################################
# Class definition of a Dog object that encapsulates all important 
# attributes and methods of a dog. 
# Author:  
#########################################################################################
############################## Option 3 #############################################


In [33]:
qaic_sess.reset()